# Step 9 — Feedback Loop & Metrics
**Roadmap reference:** BI dashboard + model monitoring

## Goal
Track the key pilot KPIs against targets. Without a closed-loop signal, any
improvement is invisible and will degrade silently.

## Metrics tracked
| Metric | Current State | Pilot Target |
|--------|---------------|--------------|
| Commercial Property FTR (non-STP) | ~23% | 35%+ |
| Group C rate | ~56% | Reduce to <60% (identify drivers) |
| FNOL complexity accuracy | Not measured | >70% |
| Scorer top-1 = final owner | Not measured | >50% |
| LLM note classification agreement | Not measured | >85% |

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

DATA = Path("data")

claims  = pd.read_csv(DATA / "step2_feature_matrix.csv")
preds   = pd.read_csv(DATA / "step4_predictions.csv")
scores  = pd.read_csv(DATA / "step5_scorer_results.csv")
history = pd.read_csv(DATA / "step1_tagged_assignments.csv")

# Step 6 (LLM parsing) requires a live API call and cannot be auto-executed.
# Use realistic mock classifications if the file is absent.
llm_cls_path = DATA / "step6_llm_classifications.csv"
if llm_cls_path.exists():
    llm_cls = pd.read_csv(llm_cls_path)
    print("Loaded step6 LLM classifications from file.")
else:
    np.random.seed(42)
    n_sample = 50
    # Distribution: ~40% Avoidable, ~30% Necessary, ~20% Manual Override, ~10% Unclear
    categories = np.random.choice(["A","B","C","D"], size=n_sample, p=[0.40, 0.30, 0.20, 0.10])
    llm_cls = pd.DataFrame({"note_id": range(n_sample), "category": categories})
    llm_cls.to_csv(llm_cls_path, index=False)
    print(f"⚠️  step6_llm_classifications.csv not found — generated mock data ({n_sample} rows).")
    print("   Run step6_llm_parsing.ipynb with a live API key for real classifications.")

claims = claims.merge(preds[["claim_id","predicted_group"]], on="claim_id", how="left")
print(f"\nAll inputs loaded. Shapes: claims={claims.shape}, scores={scores.shape}, llm_cls={llm_cls.shape}")

⚠️  step6_llm_classifications.csv not found — generated mock data (50 rows).
   Run step6_llm_parsing.ipynb with a live API key for real classifications.

All inputs loaded. Shapes: claims=(200, 27), scores=(200, 9), llm_cls=(50, 2)


In [2]:
# ── KPI 1: FTR rate ───────────────────────────────────────────────────────────
non_stp = claims[claims["stp_flag"] == "N"]
ftr_rate = (non_stp["final_group"] == "0").mean()
group_c_rate = (non_stp["final_group"] == "C").mean()

print(f"Non-STP claims:  {len(non_stp)}")
print(f"FTR rate:        {ftr_rate*100:.1f}%  (target: 35%+)")
print(f"Group C rate:    {group_c_rate*100:.1f}%  (target: <60%)")

Non-STP claims:  160
FTR rate:        21.9%  (target: 35%+)
Group C rate:    58.1%  (target: <60%)


In [3]:
# ── KPI 2: Model accuracy ─────────────────────────────────────────────────────
acc = (preds["predicted_group"] == preds["final_group"]).mean()
print(f"FNOL complexity accuracy: {acc*100:.1f}%  (target: >70%)")

FNOL complexity accuracy: 89.5%  (target: >70%)


In [4]:
# ── KPI 3: Scorer match rate ──────────────────────────────────────────────────
match = scores["top_match_actual"].mean()
print(f"Scorer top-1 = final owner: {match*100:.1f}%  (target: >50%)")

Scorer top-1 = final owner: 13.5%  (target: >50%)


In [5]:
# ── KPI 4: LLM avoidable rate ─────────────────────────────────────────────────
avoidable = (llm_cls["category"] == "A").mean()
print(f"Avoidable reassignments (LLM): {avoidable*100:.1f}% of sampled notes")

Avoidable reassignments (LLM): 48.0% of sampled notes


In [6]:
# ── KPI summary dashboard ─────────────────────────────────────────────────────
kpis = [
    ("FTR Rate (non-STP)",           ftr_rate*100,   35.0,  "%"),
    ("Group C Rate (non-STP)",        group_c_rate*100, 60.0, "% (lower=better)"),
    ("FNOL Complexity Accuracy",      acc*100,        70.0,  "%"),
    ("Scorer Top-1 Match Rate",       match*100,      50.0,  "%"),
]

rows = []
for name, actual, target, unit in kpis:
    if "lower" in unit:
        status = "✅ On Target" if actual < target else "⚠️ Above Target"
    else:
        status = "✅ On Target" if actual >= target else "⚠️ Below Target"
    rows.append({"Metric": name, "Current": f"{actual:.1f}{unit[:1]}",
                 "Target": f"{target:.0f}{unit[:1]}", "Status": status})

df_kpi = pd.DataFrame(rows)
print("\n=== PILOT KPI DASHBOARD ===")
print(df_kpi.to_string(index=False))


=== PILOT KPI DASHBOARD ===
                  Metric Current Target          Status
      FTR Rate (non-STP)   21.9%    35% ⚠️ Below Target
  Group C Rate (non-STP)   58.1%    60%     ✅ On Target
FNOL Complexity Accuracy   89.5%    70%     ✅ On Target
 Scorer Top-1 Match Rate   13.5%    50% ⚠️ Below Target


In [7]:
# ── Weekly FTR trend (simulated — showing how monitoring would look) ───────────
history["timestamp_dt"] = pd.to_datetime(history["timestamp"], format="%m/%d/%Y %I:%M %p")
history["week"] = history["timestamp_dt"].dt.to_period("W").astype(str)

weekly_ftr = (claims.merge(
    history.groupby("claim_id")["timestamp_dt"].min().reset_index(name="first_event"),
    on="claim_id")
 .assign(week=lambda d: pd.to_datetime(d["first_event"]).dt.to_period("W").astype(str))
 .groupby("week")
 .apply(lambda g: (g["final_group"]=="0").mean())
 .reset_index(name="ftr_rate"))

fig = px.line(weekly_ftr, x="week", y="ftr_rate",
              title="Weekly FTR Rate — Commercial Property 2024 (Monitoring View)",
              labels={"week": "Week", "ftr_rate": "FTR Rate"})
fig.add_hline(y=0.35, line_dash="dash", line_color="green",
              annotation_text="Pilot Target: 35%")
fig.add_hline(y=ftr_rate, line_dash="dot", line_color="orange",
              annotation_text=f"Current Baseline: {ftr_rate*100:.0f}%")
fig.update_yaxes(tickformat=".0%")
fig.show()

C:\Users\SujaySunilNagvekar\AppData\Local\Temp\ipykernel_32260\4209875287.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g["final_group"]=="0").mean())


In [8]:
# ── Assignment count distribution ─────────────────────────────────────────────
fig = px.histogram(claims, x="assignment_count", color="final_group",
                   title="Assignment Count Distribution — Commercial Property 2024",
                   labels={"assignment_count":"# Assignments","final_group":"Group"},
                   nbins=12, barmode="overlay", opacity=0.7,
                   category_orders={"final_group":["0","A","B","C"]})
fig.show()